# 02c — Import Fidelity CSV Exports to Supabase

Use this notebook when you have Fidelity CSV files but Plaid is not yet connected.

**Exports needed from Fidelity:**
- **Portfolio Positions** → Accounts & Trade → Portfolio → Download
- **Activity Orders History** → Accounts & Trade → Activity & Orders → Download

Place the files in `agent/csv_import/` and update the paths below.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

# Add csv_import to path
NOTEBOOK_DIR = Path().resolve()
AGENT_DIR = NOTEBOOK_DIR.parent
CSV_IMPORT_DIR = AGENT_DIR / 'csv_import'
sys.path.insert(0, str(CSV_IMPORT_DIR))
sys.path.insert(0, str(AGENT_DIR))

load_dotenv(AGENT_DIR / '.env')

# ── Configure file paths here ─────────────────────────────────────────────────
HOLDINGS_FILE = CSV_IMPORT_DIR / 'samples' / 'sample_holdings.csv'      # ← change me
TRANSACTIONS_FILE = CSV_IMPORT_DIR / 'samples' / 'sample_transactions.csv'  # ← change me

print(f'Holdings file:     {HOLDINGS_FILE}')
print(f'Transactions file: {TRANSACTIONS_FILE}')
print(f'Holdings exists:     {HOLDINGS_FILE.exists()}')
print(f'Transactions exists: {TRANSACTIONS_FILE.exists()}')

In [ ]:
# ── Parse Holdings ────────────────────────────────────────────────────────────
from parse_holdings import parse_holdings_csv

holdings = []
if HOLDINGS_FILE.exists():
    holdings = parse_holdings_csv(HOLDINGS_FILE)
    df_h = pd.DataFrame(holdings)
    print(f'{len(holdings)} holdings parsed')
    display(df_h[['ticker', 'type', 'account_type', 'quantity', 'price', 'value', 'gain_loss', 'cost_basis']].head(20))
else:
    print('Holdings file not found — skipping')

In [ ]:
# ── Parse Transactions ────────────────────────────────────────────────────────
from parse_transactions import parse_transactions_csv

transactions = []
skipped_actions = []
if TRANSACTIONS_FILE.exists():
    transactions, skipped_actions = parse_transactions_csv(TRANSACTIONS_FILE)
    df_t = pd.DataFrame(transactions)
    print(f'{len(transactions)} transactions parsed')
    if skipped_actions:
        print(f'  ⚠  Unrecognized actions: {set(skipped_actions)}')
    display(df_t[['date', 'type', 'ticker', 'quantity', 'price', 'amount', 'fees']].head(20))
else:
    print('Transactions file not found — skipping')

In [ ]:
# ── FIFO Realized Gains Preview ───────────────────────────────────────────────
from parse_transactions import compute_fifo_gains

gains = []
if transactions:
    gains = compute_fifo_gains(transactions)
    df_g = pd.DataFrame(gains) if gains else pd.DataFrame()
    print(f'{len(gains)} realized gains computed (FIFO)')
    if not df_g.empty:
        display(df_g[['ticker', 'sell_date', 'quantity', 'proceeds', 'cost_basis', 'fees', 'gain_loss', 'short_term']])
    else:
        print('No sell transactions found — no gains to compute')
else:
    print('No transactions parsed — skipping gains computation')

In [ ]:
# ── Pre-flight Check ──────────────────────────────────────────────────────────
from collections import Counter

print('Pre-flight summary')
print('=' * 50)

if holdings:
    total_value = sum(h.get('value') or 0 for h in holdings)
    accounts = {h['account_id'] for h in holdings}
    print(f'\nHoldings:  {len(holdings)} rows')
    print(f'  Accounts:    {accounts}')
    print(f'  Total value: ${total_value:,.2f}')
    print(f'  Types: {dict(Counter(h["type"] for h in holdings))}')

if transactions:
    dates = [t['date'] for t in transactions if t.get('date')]
    print(f'\nTransactions: {len(transactions)} rows')
    print(f'  Date range: {min(dates)} → {max(dates)}')
    print(f'  Types: {dict(Counter(t["type"] for t in transactions))}')

if gains:
    total_gain = sum(g.get('gain_loss') or 0 for g in gains)
    print(f'\nRealized gains: {len(gains)} entries')
    print(f'  Total gain/loss: ${total_gain:,.2f}')

print('\n' + '=' * 50)
print('Run the next cell to sync to Supabase.')

In [ ]:
# ── Sync to Supabase ──────────────────────────────────────────────────────────
# No CLI confirmation needed in the notebook — interactive context provides approval.
import math
from datetime import datetime
from supabase import create_client
import sys
sys.path.insert(0, str(AGENT_DIR))
from sync_to_supabase import (
    clean,
    compute_brokerage_summary,
    compute_asset_type_summary,
    compute_account_category_summary,
)

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_SERVICE_ROLE_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise EnvironmentError('Missing SUPABASE_URL or SUPABASE_SERVICE_ROLE_KEY in .env')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
now_iso = datetime.utcnow().isoformat() + 'Z'
print(f'Connected to Supabase: {SUPABASE_URL[:40]}…')

def batch(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# 1. pipeline_run
run_res = supabase.table('pipeline_runs').insert(
    {'trigger': 'csv', 'status': 'success', 'run_at': now_iso}
).execute()
pipeline_run_id = run_res.data[0]['id']
print(f'✅ pipeline_runs: 1 row (id={pipeline_run_id})')

# 2. snapshot + holdings
snapshot_id = None
if holdings:
    total_value = sum(h.get('value') or 0 for h in holdings)
    total_cost_basis = sum(h.get('cost_basis') or 0 for h in holdings if h.get('cost_basis'))
    total_gain_loss = sum(h.get('gain_loss') or 0 for h in holdings if h.get('gain_loss'))

    snap_res = supabase.table('portfolio_snapshots').insert(clean({
        'pipeline_run_id': pipeline_run_id,
        'total_value': round(total_value, 2),
        'total_cost_basis': round(total_cost_basis, 2),
        'total_gain_loss': round(total_gain_loss, 2),
        'total_positions': len(holdings),
        'brokerages_json': compute_brokerage_summary(holdings),
        'asset_types_json': compute_asset_type_summary(holdings, total_value),
        'account_categories_json': compute_account_category_summary(holdings, total_value),
        'created_at': now_iso,
    })).execute()
    snapshot_id = snap_res.data[0]['id']
    print(f'✅ portfolio_snapshots: 1 row (id={snapshot_id})')

    for h in holdings:
        h['snapshot_id'] = snapshot_id

    total_inserted = 0
    for chunk in batch(holdings, 100):
        res = supabase.table('holdings').insert(clean(chunk)).execute()
        total_inserted += len(res.data)
    print(f'✅ holdings: {total_inserted} rows')

# 3. transactions
if transactions:
    total_inserted = 0
    for chunk in batch(transactions, 100):
        res = supabase.table('transactions').upsert(clean(chunk), on_conflict='plaid_transaction_id').execute()
        total_inserted += len(res.data)
    print(f'✅ transactions: {total_inserted} rows')

# 4. realized_gains
if gains:
    total_inserted = 0
    for chunk in batch(gains, 100):
        res = supabase.table('realized_gains').insert(clean(chunk)).execute()
        total_inserted += len(res.data)
    print(f'✅ realized_gains: {total_inserted} rows')

print('\n🎉 Import complete!')

In [ ]:
# ── Verify Row Counts ─────────────────────────────────────────────────────────
print('Verifying CSV-sourced rows in Supabase:')

for table in ['holdings', 'transactions', 'realized_gains']:
    res = supabase.table(table).select('id', count='exact').eq('source', 'csv').execute()
    print(f'  {table:<20s}: {res.count} rows (source=csv)')

res = supabase.table('pipeline_runs').select('id', count='exact').eq('trigger', 'csv').execute()
print(f'  pipeline_runs       : {res.count} rows (trigger=csv)')